In [4]:
# Set up all required functions
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [5]:
# Fresh kernel — Cell 1
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

In [ ]:
spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .getOrCreate()

print("Spark version:", spark.version)  # confirm JVM is alive

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0


In [7]:
df_read_review_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_review.json")

In [8]:
df_review_text = df_read_review_data.select(F.col("review_id"),F.col("text"))

In [ ]:
df_review_text.show()

In [23]:
review = df_review_text.withColumn(
    "text_clean",
    F.lower(F.trim(F.regexp_replace("text", r"\s+", " ")))
).drop(F.col("text"))

review = review.withColumn("text_clean", F.regexp_replace("text_clean", r"([a-z])\1{2,}", "$1$1"))
review = review.withColumn("text_clean", F.regexp_replace("text_clean", r"([!?.,*~-])\1{2,}", "$1"))

In [24]:
df_clean = review.filter(F.col("text_clean").rlike(r"[a-zA-Z0-9]"))

In [25]:
df_surrogate_key = df_clean.withColumn("review_pk_sk", F.monotonically_increasing_id() + 1)

In [28]:
df_final = df_surrogate_key.select(F.col("review_pk_sk"),F.col("review_id"),F.col("text_clean").alias("review"))    

In [30]:
df_final.write.mode("overwrite").parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_review")